In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

print("Environnement prêt")

Environnement prêt


In [44]:
df = pd.read_csv("../data/raw/bluebook-for-bulldozers/TrainAndValid.csv",
                 low_memory=False)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [56]:
df["SalesID"].nunique()
df.shape[0]

412698

Unité d'observation

Une ligne correspond à une transaction de vente aux enchères d'un engin.
'SalesID' est unique (412 698 valeurs distinctes = 412 698 lignes).

In [ ]:
df["MachineID"].nunique()


In [ ]:
df["MachineID"].value_counts().head(100)


In [73]:
counts = df["MachineID"].value_counts()


In [ ]:
(counts > 1).sum()/len(counts)

Identifiants

- 'SalesID' est unique : une ligne = une transaction.
- 348 808 machines distinctes pour 412 698 transactions.
- 46 736 machines (13%) apparaissent plusieurs fois.
- Risque de fuite limité par le split temporel : on entraîne sur le passé, on teste sur le futur.

In [ ]:
df["saledate"] = pd.to_datetime(df["saledate"])


In [ ]:
df["saledate"].dt.year
df["saledate"].dt.month
print(df["saledate"].min())
print(df["saledate"].max())
print(df["saledate"].dt.year.value_counts().sort_index())

In [ ]:
df.groupby("ProductGroup")["MachineHoursCurrentMeter"] \
  .apply(lambda x: x.isnull().mean() * 100) \
  .sort_values(ascending=False)

Le taux de valeurs manquantes de `MachineHoursCurrentMeter` est proche dans tous les groupes de produits. L’absence ne semble donc pas spécifique à une catégorie d’engins et paraît davantage liée à la collecte des données.
MachineHoursCurrentMeter

Selon le dictionnaire, une valeur nulle ou égale à 0 signifie "non déclaré".
- 265 194 valeurs manquantes (64%)
- 73 834 valeurs égales à zéro à traiter comme manquantes
- Décision : convertir les zéros en NaN dans le pipeline
- Créer un indicateur binaire `hours_reported` pour capter l'information de disponibilité